# Atelier Scikit-learn

## Partie 0 – mise en place de l’environnement

### Installer et importer seaborn, matplotlib et pandas 

In [29]:
import seaborn as sns
import pandas as pd
import matplotlib as plt

print("Seaborn version :", sns.__version__)
print("Pandas version  :", pd.__version__)
print("matplotlib version  :", plt.__version__)

Seaborn version : 0.13.2
Pandas version  : 3.0.5
matplotlib version  : 3.11.1


### Importer mesures_capteurs.csv dans le dataframe df 

In [30]:
df =pd.read_csv("data/mesures_capteurs.csv")

### Explorer le dataframe df 

In [31]:
print("Premières lignes :")
display(df.head())
print("\nDimensions :", df.shape)

Premières lignes :


,id_mesure,date_heure,id_capteur,batiment,temperature,humidite,pression,consommation,etat
0,M0413,2026-01-22 04:00:00,C005,B002,25.46,58.06,1008.95,287.28,OK
1,M0290,2026-01-17 01:00:00,C002,B001,24.00,79.73,993.39,116.20,OK
2,M0077,2026-01-08 04:00:00,C005,B002,25.82,54.47,1010.32,288.50,OK
3,M0079,2026-01-08 06:00:00,C007,B003,28.23,69.39,1019.62,136.65,OK
4,M0183,2026-01-12 14:00:00,C003,B001,20.58,53.80,1016.58,182.62,OK



Dimensions : (605, 9)


## Partie 1 – Gestion des doublons 
## Avec Pandas, 
### 1) vérifier l’existence de doublons dans df 

In [32]:
doublons= df.duplicated().sum()
print(f"le dataframe compte {doublons} doublons")

le dataframe compte 5 doublons


### 2) le cas échéant, supprimer les doublons puis vérifier la suppression  

In [33]:
import pandas as pd
#affichage avant suppression des doublons
doublons= df.duplicated().sum()
print("Avant suppression des doublons")
print(f"le dataframe compte {doublons} doublons")
#suppression des doublons 
df=df.drop_duplicates()
doublons= df.duplicated().sum()
print("Apres suppression des doublons")
print(f"le dataframe compte {doublons} doublons")


Avant suppression des doublons
le dataframe compte 5 doublons
Apres suppression des doublons
le dataframe compte 0 doublons


## Partie 2 – Sélection de y (cible) et X (caractéristiques) 
### 1) Définir "etat" comme la cible ou valeur à prédire et "temperature", "humidite", "pression" et "consommation" comme caractéristiques ou variables explicatives 

In [34]:
# Vérification des valeurs manquantes dans la colonne cible 'etat'
nan_etat = df['etat'].isnull().sum()
print(f"Nombre de valeurs manquantes dans la variable cible 'etat' : {nan_etat}")

# Suppression des lignes où la variable cible 'etat' est manquante
if nan_etat > 0:
    df = df.dropna(subset=['etat'])
    print("Lignes avec variable cible manquante supprimées !")

print(f"Dimensions du DataFrame df après nettoyage : {df.shape[0]} lignes.")

#  Définir 'etat' comme la cible ou valeur à prédire et 'temperature', 'humidite', 'pression' et 'consommation' comme caractéristiques
X = df[["temperature", "humidite", "pression", "consommation"]]
y = df["etat"]

Nombre de valeurs manquantes dans la variable cible 'etat' : 4
Lignes avec variable cible manquante supprimées !
Dimensions du DataFrame df après nettoyage : 596 lignes.


### 2) Afficher les cinq premières lignes de X et de y


In [35]:
print("--- 5 premières lignes de X (Caractéristiques ou variables explicatives) ---")
display(X.head())
print("\n--- 5 premières lignes de y (Variable cible ou étiquette) ---")
display(y.head())

--- 5 premières lignes de X (Caractéristiques ou variables explicatives) ---


,temperature,humidite,pression,consommation
0,25.46,58.06,1008.95,287.28
1,24.00,79.73,993.39,116.20
2,25.82,54.47,1010.32,288.50
3,28.23,69.39,1019.62,136.65
4,20.58,53.80,1016.58,182.62



--- 5 premières lignes de y (Variable cible ou étiquette) ---


0    OK
1    OK
2    OK
3    OK
4    OK
Name: etat, dtype: str

### 3) Quel est le type du problème de machine learning ? 


Il s'agit d'un problème de **classification multiclasse** (qui relève de l'apprentissage supervisé).

* **Pourquoi de l'apprentissage supervisé ?** Car nous disposons de données étiquetées : chaque mesure physique est associée à sa "bonne réponse" stockée dans la variable cible `etat` (OK, ALERTE, ERREUR).
* **Pourquoi de la classification ?** Car la variable à prédire est **qualitative/catégorielle** (l'état prend des modalités textuelles définies : OK, ALERTE, ERREUR), par opposition à un problème de régression où la variable cible est numérique continue (comme prédire un prix ou une température) .
* **Pourquoi multiclasse ?** Car la variable cible comporte **plus de deux classes** (ici 3 classes distinctes), par opposition à une classification binaire qui n'en comporte que deux.

## Partie 3 – Découpage Train/Test 

In [36]:
from sklearn.model_selection import train_test_split

# Découpage en ensembles d'entraînement (80%) et de test (20%)
X_train, X_test, y_train, y_test = train_test_split(
    X, 
    y, 
    test_size=0.2, # 20% des données serviront au test
    random_state=42, # Garantir la reproductibilité
    stratify=y     # Conserver les mêmes proportions de classes
)

print(f"Dimensions de X_train (Entraînement) : {X_train.shape[0]} lignes, {X_train.shape[1]} caractéristiques")
print(f"Dimensions de X_test (Test)          : {X_test.shape[0]} lignes, {X_test.shape[1]} caractéristiques")


Dimensions de X_train (Entraînement) : 476 lignes, 4 caractéristiques
Dimensions de X_test (Test)          : 120 lignes, 4 caractéristiques


In [37]:
from collections import Counter

# Vérification de la répartition des classes dans les deux ensembles
print("Répartition des classes dans l'ensemble d'entraînement (y_train) :")
for classe, count in Counter(y_train).items():
    print(f" - {classe:7s} : {count:3d} exemples ({count/len(y_train)*100:.2f}%)")

print("\nRépartition des classes dans l'ensemble de test (y_test) :")
for classe, count in Counter(y_test).items():
    print(f" - {classe:7s} : {count:3d} exemples ({count/len(y_test)*100:.2f}%)")


Répartition des classes dans l'ensemble d'entraînement (y_train) :
 - OK      : 449 exemples (94.33%)
 - ALERTE  :  23 exemples (4.83%)
 - ERREUR  :   4 exemples (0.84%)

Répartition des classes dans l'ensemble de test (y_test) :
 - OK      : 113 exemples (94.17%)
 - ALERTE  :   6 exemples (5.00%)
 - ERREUR  :   1 exemples (0.83%)


## Partie 4 – Gestion des valeurs manquantes 
### 1) Vérifier l’existence de valeurs manquantes 

In [38]:
print("--- Valeurs manquantes dans X_train ---")
print(X_train.isnull().sum())
print("\n--- Valeurs manquantes dans X_test ---")
print(X_test.isnull().sum())

--- Valeurs manquantes dans X_train ---
temperature     5
humidite        4
pression        5
consommation    3
dtype: int64

--- Valeurs manquantes dans X_test ---
temperature     1
humidite        1
pression        0
consommation    2
dtype: int64


### 2) Sélectionner SimpleImputer avec la médiane 

In [39]:
from sklearn.impute import SimpleImputer
imputer = SimpleImputer(strategy="median")


### 3) Qu’est ce qui justifie le choix de la médiane ? 

Le choix de la **médiane** (strategy="median") pour remplacer les valeurs manquantes est justifié par sa **robustesse face aux valeurs aberrantes ou extrêmes** (outliers).
Contrairement à la moyenne (qui est fortement influencée par des anomalies, par exemple si un capteur de température défectueux renvoie temporairement 150°C ou si un pic anormal est mesuré), la médiane représente la valeur centrale du jeu de données et reste stable même en présence d'anomalies de mesure. C'est l'imputation par défaut recommandée en milieu industriel et IoT pour préserver la cohérence des distributions.


### 4) Trouver les paramètres (médianes) de l’imputeur sur X_train 

In [40]:
#On calcule la statistique sur X_train uniquement, et on l'applique sur X_train et X_test.
imputer.fit(X_train)

print("Médianes calculées par variable sur l'ensemble d'entraînement :")
for col, med in zip(X_train.columns, imputer.statistics_):
    print(f" - {col:12s} : {med:.2f}")

Médianes calculées par variable sur l'ensemble d'entraînement :
 - temperature  : 24.90
 - humidite     : 65.38
 - pression     : 1012.30
 - consommation : 206.59


### 5) Déterminer X_train_imputed et X_test_imputed, les transformés de X_train et X_test

In [42]:
import numpy as np

X_train_imputed = imputer.transform(X_train)
X_test_imputed = imputer.transform(X_test)
# Vérification finale de l'absence de valeurs manquantes
print(f"Valeurs manquantes restantes dans X_train_imputed : {np.isnan(X_train_imputed).sum()}")
print(f"Valeurs manquantes restantes dans X_test_imputed  : {np.isnan(X_test_imputed).sum()}")


Valeurs manquantes restantes dans X_train_imputed : 0
Valeurs manquantes restantes dans X_test_imputed  : 0


## Partie 5 – Mise à l'échelle 
### 1) Sélectionner StandardScaler pour mettre à l’échelle les transformés de l’imputation 

In [43]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()